# Preprocessing Dataset Resep — Skripsi Chatbot Rekomendasi Resep (NLP)

Notebook ini menyiapkan data resep sebelum masuk ke tahap embedding (SBERT), indexing (ChromaDB), dan reranking berbasis efisiensi bahan.

**Yang dihasilkan di akhir notebook ini:**
- `title` — judul resep (bersih)
- `ingredients_clean` — list bahan mentah yang sudah dibersihkan (untuk ditampilkan ke user)
- `ingredients_text` — gabungan bahan jadi 1 string (dipakai buat embedding)
- `ingredients_parsed` — list dict `{raw, quantity, unit, name}` per bahan (dipakai buat matching & skor efisiensi bahan)
- `ingredients_name_only` — list nama bahan saja, sudah lowercase & dinormalisasi (dipakai buat matching cepat dengan input user)
- `steps` — list langkah masak (bersih)
- `embedding_text` — teks final yang akan di-encode SBERT

> Catatan: parsing bahan pakai pendekatan regex (bukan NER/LLM) supaya cepat dan tidak butuh model tambahan. Ini **tidak akan 100% akurat** untuk semua format penulisan bahan (bahasa Indonesia resep itu sangat bervariasi), jadi di bagian akhir ada sel pengecekan hasil parsing — kirim hasilnya ke saya kalau ada pola yang kelewat, nanti kita perbaiki regexnya bareng.

In [1]:
import pandas as pd
import numpy as np
import re
import ast
import unicodedata
from pathlib import Path

pd.set_option("display.max_colwidth", 150)

## 1. Load Dataset

In [2]:
DATA_PATH = "dataset_resep.parquet"

df = pd.read_parquet(DATA_PATH)
print("Shape:", df.shape)
df.head(1)

Shape: (66419, 6)


,ingredients,title,steps,num_ingredients,num_steps,char_count
0,"[1 ekor ikan bandeng, 4 buah tahu, 1 buah tomat, 1 buah wortel, 1 sdm kecap manis, Soun (saya tidak pakai), 5 siung bawang merah, 4 siung bawang p...",Semur Ikan Bandeng khas Surabaya (kids friendly),"[Goreng tahu dan ikan bandeng. (Utk ikan bandeng saya lumuri dulu dgn bumbu ikan goreng), Tumis bumbu halus hingga harum, lalu masukkan tahu dan i...",13,3,605


## 2. Eksplorasi Awal
Cek missing value, duplikat, dan tipe data kolom `ingredients` / `steps`.

In [3]:
print(df.info())
print("\nMissing value per kolom:")
print(df.isnull().sum())
print("\nJumlah duplikat berdasarkan title:", df.duplicated(subset=["title"]).sum())

sample_val = df["ingredients"].iloc[0]
print("\nTipe data kolom 'ingredients':", type(sample_val))
print("Contoh isi:", sample_val[:3] if hasattr(sample_val, "__len__") else sample_val)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66419 entries, 0 to 66418
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   ingredients      66419 non-null  object
 1   title            66419 non-null  object
 2   steps            66419 non-null  object
 3   num_ingredients  66419 non-null  int64 
 4   num_steps        66419 non-null  int64 
 5   char_count       66419 non-null  int64 
dtypes: int64(3), object(3)
memory usage: 3.0+ MB
None

Missing value per kolom:
ingredients        0
title              0
steps              0
num_ingredients    0
num_steps          0
char_count         0
dtype: int64

Jumlah duplikat berdasarkan title: 17957

Tipe data kolom 'ingredients': <class 'numpy.ndarray'>
Contoh isi: ['1 ekor ikan bandeng' '4 buah tahu' '1 buah tomat']


## 3. Pastikan `ingredients` & `steps` Berbentuk List Python
Kadang saat baca dari parquet/csv, kolom list bisa kebaca sebagai numpy array, list biasa, atau (kalau sempat lewat csv) string representasi list seperti `"['a', 'b']"`. Fungsi di bawah menangani semua kemungkinan itu.

In [4]:
def ensure_list(x):
    if isinstance(x, (list, np.ndarray)):
        return list(x)
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, (list, np.ndarray)):
                return list(parsed)
        except (ValueError, SyntaxError):
            pass
        return [x]
    return []


df["ingredients"] = df["ingredients"].apply(ensure_list)
df["steps"] = df["steps"].apply(ensure_list)

## 4. Buang Data Kosong / Tidak Valid & Duplikat

In [5]:
before = len(df)

df = df[df["ingredients"].apply(len) > 0]
df = df[df["steps"].apply(len) > 0]
df = df[df["title"].astype(str).str.strip() != ""]
df = df.drop_duplicates(subset=["title", "num_ingredients", "num_steps"]).reset_index(
    drop=True
)

after = len(df)
print(f"Baris sebelum: {before} | sesudah: {after} | dibuang: {before - after}")

Baris sebelum: 66419 | sesudah: 63685 | dibuang: 2734


## 5. Sampling 5000 Resep
Pakai `random_state` supaya hasil sampling konsisten tiap kali dijalankan ulang (reproducible untuk keperluan eksperimen skripsi).

In [6]:
N_SAMPLE = 5000
RANDOM_STATE = 42

df_sample = df.sample(n=min(N_SAMPLE, len(df)), random_state=RANDOM_STATE).reset_index(
    drop=True
)
print(df_sample.shape)
df_sample.head(3)

(5000, 6)


,ingredients,title,steps,num_ingredients,num_steps,char_count
0,"[1 ekor Baby Tuna, 1 sdm Asam Jawa, 1 sdt Garam, 1/2 sdt Kunyit Bubuk, 500 ml Air, 1 lembar Daun Salam, 1 lembar Daun Jeruk, 1 genggam Daun Kemang...",Ikan Suwir Pedas,"[Bersihkan ikan, campur dengan air, asam jawa, garam dan kunyit, kemudian masak hingga ikan matang., Ambil bagian dagingnya kemudian suwir suwir, ...",16,5,696
1,"[100 g kulit ayam, 12 siung bawang merah, 4 siung bawang putih, 250 ml minyak goreng]",Minyak Ayam Bawang,"[Rebus kulit ayam hingga air mendidih, angkat., Blender kulit ayam rebus, bawang merah, dan bawang putih. Tambahkan sedikit minyak sebagai pelarut...",4,4,387
2,"[2 butir telur ayam, 2 buah rolade, 1 tahu tuna, 2 sdm tepung serba guna, 1/4 sdt garam, secukupnya Minyak goreng, Bahan saos:, 1 siung bawang put...",Puyunghai rolade tahu tuna,"[Potong dadu rolade dan tahu tuna., Campurkan semua bahan lalu kocok., Setelah siap, kemudian goreng dengan api kecil., Buat saos, tumis bawang pu...",12,5,398


## 6. Cleaning Teks Umum
Dipakai untuk `title` dan tiap elemen di `steps` dan `ingredients` (whitespace berlebih, karakter unicode aneh, dsb).

In [7]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u200b", " ").replace(
        "\xa0", " "
    )  # zero-width space & non-breaking space
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


df_sample["title_clean"] = df_sample["title"].apply(clean_text)
df_sample["steps_clean"] = df_sample["steps"].apply(
    lambda steps: [clean_text(s) for s in steps]
)
df_sample["ingredients_clean"] = df_sample["ingredients"].apply(
    lambda lst: [clean_text(i) for i in lst]
)

## 7. Parsing Bahan (Quantity / Unit / Nama)
Ini bagian paling penting buat kebutuhan reranking nanti. Tiap string bahan mentah seperti `"250 gr udang"` atau `"2 lembar daun jeruk"` dipecah jadi:
- `quantity`: `"250"` / `"2"` (bisa `None`, atau `"secukupnya"` kalau memang ditulis begitu)
- `unit`: `"gr"` / `"lembar"`
- `name`: `"udang"` / `"daun jeruk"` — inilah yang nanti dipakai buat mencocokkan dengan bahan yang diketik user

Catatan isi dalam kurung, misal `"(potong fillet)"`, dibuang dari `name` karena itu instruksi pengolahan, bukan nama bahan.

**v2 — fix dari hasil review sebelumnya:**
1. Kata `"secukupnya"` sekarang ditangani khusus (sebelumnya pecah salah jadi `quantity="se"`, `name="cukupnya minyak goreng"`).
2. Kata majemuk `"se-" + satuan` (`sebutir`, `seekor`, `seruas`, dst) sekarang dikenali sebagai `quantity=1` + unit yang sesuai.
3. Baris header sub-bagian seperti `"Bumbu halus:"` / `"Bahan saos:"` (biasanya diakhiri `:`) di-drop dari `ingredients_parsed`/`ingredients_name_only` — itu bukan bahan, cuma label pengelompokan.
4. Unit ditambahkan lookahead supaya tidak 'memakan' awal kata bahan (sebelumnya unit `"g"` tidak dikenali sama sekali dan malah nyangkut jadi bagian nama bahan, mis. `"g  bawang bombay"`).

In [8]:
UNITS = [
    "kilogram",
    "kg",
    "gram",
    "grm",
    "gr",
    "g",
    "ons",
    "liter",
    "ltr",
    "sendok makan",
    "sdm",
    "sendok teh",
    "sdt",
    "ml",
    "l",
    "butir",
    "btr",
    "buah",
    "bh",
    "ekor",
    "siung",
    "lembar",
    "lbr",
    "batang",
    "btg",
    "ruas",
    "genggam",
    "ikat",
    "bungkus",
    "bgks",
    "papan",
    "potong",
    "ptg",
    "iris",
    "biji",
    "cm",
    "lonjor",
    "sisir",
    "sachet",
    "gelas",
    "cangkir",
    "pcs",
]
# urutkan dari yang paling panjang supaya regex match unit multi-kata ("sendok makan") duluan sebelum "sdm"
UNITS_SORTED = sorted(UNITS, key=len, reverse=True)
UNIT_PATTERN = "|".join(re.escape(u) for u in UNITS_SORTED)

QTY_PATTERN = r"(\d+[\.,]?\d*(?:\s*[/-]\s*\d+)?|setengah|seperempat|sepertiga)"

# unit harus diikuti karakter BUKAN huruf (spasi/akhir kalimat/tanda baca),
# supaya "g" tidak ikut memakan awal kata seperti "garam"/"gula"
INGREDIENT_RE = re.compile(
    rf"^\s*(?P<qty>{QTY_PATTERN})?\s*(?P<unit>(?:{UNIT_PATTERN})(?![a-zA-Z]))?\.?\s*(?P<name>.*)$",
    flags=re.IGNORECASE,
)

PAREN_RE = re.compile(r"\(.*?\)")

# kata majemuk "se-" + satuan -> quantity 1 + unit yang sesuai
SE_UNIT_MAP = {
    "sesendok": ("1", "sdm"),
    "secangkir": ("1", "cangkir"),
    "sebutir": ("1", "butir"),
    "seekor": ("1", "ekor"),
    "sesiung": ("1", "siung"),
    "selembar": ("1", "lembar"),
    "seruas": ("1", "ruas"),
    "segenggam": ("1", "genggam"),
    "sebungkus": ("1", "bungkus"),
    "sepotong": ("1", "potong"),
    "seiris": ("1", "iris"),
    "sebuah": ("1", "buah"),
    "sebatang": ("1", "batang"),
    "sesendokteh": ("1", "sdt"),
}
SE_UNIT_RE = re.compile(
    r"^(" + "|".join(sorted(SE_UNIT_MAP.keys(), key=len, reverse=True)) + r")\b",
    flags=re.IGNORECASE,
)
SECUKUPNYA_RE = re.compile(r"^secukupnya\b", flags=re.IGNORECASE)


def is_section_header(raw: str) -> bool:
    """Deteksi baris label sub-bagian, mis. 'Bumbu halus:', 'Bahan saos:', bukan bahan asli."""
    text = clean_text(raw)
    stripped = re.sub(r"^[^\w]+", "", text).strip()  # buang emoji/simbol di awal
    return stripped.endswith(":")


def parse_ingredient(raw: str) -> dict:
    text = clean_text(raw)
    text_no_paren = re.sub(r"\s+", " ", PAREN_RE.sub("", text)).strip()

    m = SECUKUPNYA_RE.match(text_no_paren)
    if m:
        name = text_no_paren[m.end() :].strip(" ,.-")
        return {
            "raw": text,
            "quantity": "secukupnya",
            "unit": None,
            "name": (name or text_no_paren).lower().strip(),
        }

    m = SE_UNIT_RE.match(text_no_paren)
    if m:
        key = m.group(1).lower()
        qty, unit = SE_UNIT_MAP[key]
        name = text_no_paren[m.end() :].strip(" ,.-")
        return {
            "raw": text,
            "quantity": qty,
            "unit": unit,
            "name": (name or text_no_paren).lower().strip(),
        }

    match = INGREDIENT_RE.match(text_no_paren)
    if not match:
        return {
            "raw": text,
            "quantity": None,
            "unit": None,
            "name": text_no_paren.lower().strip(),
        }
    qty = match.group("qty")
    unit = match.group("unit")
    name = match.group("name").strip(" ,.-")
    if not name:
        name = text_no_paren
    return {
        "raw": text,
        "quantity": qty.strip() if qty else None,
        "unit": unit.lower().strip() if unit else None,
        "name": name.lower().strip(),
    }


def parse_ingredients_list(ingredients):
    # baris section header (mis. 'Bumbu halus:') di-skip, bukan bahan asli
    # entri sisa noise 1-karakter (mis. 'p', 's', '2' -- sisa artefak scraping di data mentah) juga di-skip
    result = []
    for i in ingredients:
        if is_section_header(i):
            continue
        parsed = parse_ingredient(i)
        if len(parsed["name"].strip()) <= 1:
            continue
        result.append(parsed)
    return result


df_sample["ingredients_parsed"] = df_sample["ingredients_clean"].apply(
    parse_ingredients_list
)
df_sample["ingredients_name_only"] = df_sample["ingredients_parsed"].apply(
    lambda lst: [item["name"] for item in lst if item["name"]]
)

### Cek Hasil Parsing
**Penting:** cek beberapa contoh di bawah ini sebelum lanjut. Kalau ada nama bahan yang masih 'kotor' (misal masih kebawa kata 'secukupnya', 'sesuai selera', dsb.), kirim contohnya ke saya biar regex-nya kita perbaiki.

In [9]:
for i in range(5):
    row = df_sample.iloc[i]
    print("Judul   :", row["title_clean"])
    print("Mentah  :", row["ingredients_clean"])
    print("Parsed  :")
    for item in row["ingredients_parsed"]:
        print("   ", item)
    print("-" * 80)

Judul   : Ikan Suwir Pedas
Mentah  : ['1 ekor Baby Tuna', '1 sdm Asam Jawa', '1 sdt Garam', '1/2 sdt Kunyit Bubuk', '500 ml Air', '1 lembar Daun Salam', '1 lembar Daun Jeruk', '1 genggam Daun Kemangi', '3 buah Bawang Merah', '2 siung Bawang Putih', '5 biji Cabe Merah Besar', '10 biji Cabe Rawit', '1 ruas Jahe', '1 batang Serai', '1 sdt Garam', '1/2 sdt Kaldu Bubuk']
Parsed  :
    {'raw': '1 ekor Baby Tuna', 'quantity': '1', 'unit': 'ekor', 'name': 'baby tuna'}
    {'raw': '1 sdm Asam Jawa', 'quantity': '1', 'unit': 'sdm', 'name': 'asam jawa'}
    {'raw': '1 sdt Garam', 'quantity': '1', 'unit': 'sdt', 'name': 'garam'}
    {'raw': '1/2 sdt Kunyit Bubuk', 'quantity': '1/2', 'unit': 'sdt', 'name': 'kunyit bubuk'}
    {'raw': '500 ml Air', 'quantity': '500', 'unit': 'ml', 'name': 'air'}
    {'raw': '1 lembar Daun Salam', 'quantity': '1', 'unit': 'lembar', 'name': 'daun salam'}
    {'raw': '1 lembar Daun Jeruk', 'quantity': '1', 'unit': 'lembar', 'name': 'daun jeruk'}
    {'raw': '1 genggam 

## 8. Bangun Teks Gabungan untuk Embedding (SBERT)
`ingredients_text` = bahan digabung jadi satu string.
`embedding_text` = judul + bahan, ini yang nanti di-encode SBERT dan disimpan ke ChromaDB.

In [10]:
def build_embedding_text(row) -> str:
    ingredients_str = ", ".join(row["ingredients_clean"])
    return f"{row['title_clean']}. Bahan: {ingredients_str}"


df_sample["ingredients_text"] = df_sample["ingredients_clean"].apply(
    lambda lst: ", ".join(lst)
)
df_sample["embedding_text"] = df_sample.apply(build_embedding_text, axis=1)

df_sample[["title_clean", "ingredients_text", "embedding_text"]].head(3)

,title_clean,ingredients_text,embedding_text
0,Ikan Suwir Pedas,"1 ekor Baby Tuna, 1 sdm Asam Jawa, 1 sdt Garam, 1/2 sdt Kunyit Bubuk, 500 ml Air, 1 lembar Daun Salam, 1 lembar Daun Jeruk, 1 genggam Daun Kemangi...","Ikan Suwir Pedas. Bahan: 1 ekor Baby Tuna, 1 sdm Asam Jawa, 1 sdt Garam, 1/2 sdt Kunyit Bubuk, 500 ml Air, 1 lembar Daun Salam, 1 lembar Daun Jeru..."
1,Minyak Ayam Bawang,"100 g kulit ayam, 12 siung bawang merah, 4 siung bawang putih, 250 ml minyak goreng","Minyak Ayam Bawang. Bahan: 100 g kulit ayam, 12 siung bawang merah, 4 siung bawang putih, 250 ml minyak goreng"
2,Puyunghai rolade tahu tuna,"2 butir telur ayam, 2 buah rolade, 1 tahu tuna, 2 sdm tepung serba guna, 1/4 sdt garam, secukupnya Minyak goreng, Bahan saos:, 1 siung bawang puti...","Puyunghai rolade tahu tuna. Bahan: 2 butir telur ayam, 2 buah rolade, 1 tahu tuna, 2 sdm tepung serba guna, 1/4 sdt garam, secukupnya Minyak goren..."


## 9. Cek Distribusi & Filter Outlier (Opsional)
Lihat distribusi jumlah bahan & langkah dulu sebelum memutuskan mau difilter atau tidak. Resep dengan bahan terlalu sedikit (mis. 1) atau kepanjangan bisa mengganggu kualitas retrieval.

In [11]:
print(df_sample[["num_ingredients", "num_steps", "char_count"]].describe())

# Contoh filter opsional -- sesuaikan ambang batasnya sesuai hasil describe() di atas,
# lalu un-comment kalau memang mau dipakai:
# df_sample = df_sample[(df_sample['num_ingredients'] >= 2) & (df_sample['num_ingredients'] <= 30)].reset_index(drop=True)
# print("Shape setelah filter:", df_sample.shape)

       num_ingredients    num_steps   char_count
count      5000.000000  5000.000000  5000.000000
mean         12.863000     5.180800   771.290400
std           5.290315     2.033258   345.066392
min           2.000000     2.000000   140.000000
25%           9.000000     4.000000   547.000000
50%          12.000000     5.000000   709.500000
75%          16.000000     6.000000   916.000000
max          50.000000    26.000000  3368.000000


## 10. Simpan Hasil Preprocessing

In [12]:
OUTPUT_COLUMNS = [
    "title_clean",
    "ingredients_clean",
    "ingredients_text",
    "ingredients_parsed",
    "ingredients_name_only",
    "steps_clean",
    "embedding_text",
    "num_ingredients",
    "num_steps",
    "char_count",
]

df_final = df_sample[OUTPUT_COLUMNS].reset_index(drop=True)
df_final = df_final.rename(columns={"title_clean": "title", "steps_clean": "steps"})

OUTPUT_PATH = "dataset_resep_preprocessed_5000.parquet"
df_final.to_parquet(OUTPUT_PATH, index=False)
print(f"Tersimpan di: {OUTPUT_PATH}")
print("Shape akhir:", df_final.shape)
df_final.head(3)

Tersimpan di: dataset_resep_preprocessed_5000.parquet
Shape akhir: (5000, 10)


,title,ingredients_clean,ingredients_text,ingredients_parsed,ingredients_name_only,steps,embedding_text,num_ingredients,num_steps,char_count
0,Ikan Suwir Pedas,"[1 ekor Baby Tuna, 1 sdm Asam Jawa, 1 sdt Garam, 1/2 sdt Kunyit Bubuk, 500 ml Air, 1 lembar Daun Salam, 1 lembar Daun Jeruk, 1 genggam Daun Kemang...","1 ekor Baby Tuna, 1 sdm Asam Jawa, 1 sdt Garam, 1/2 sdt Kunyit Bubuk, 500 ml Air, 1 lembar Daun Salam, 1 lembar Daun Jeruk, 1 genggam Daun Kemangi...","[{'raw': '1 ekor Baby Tuna', 'quantity': '1', 'unit': 'ekor', 'name': 'baby tuna'}, {'raw': '1 sdm Asam Jawa', 'quantity': '1', 'unit': 'sdm', 'na...","[baby tuna, asam jawa, garam, kunyit bubuk, air, daun salam, daun jeruk, daun kemangi, bawang merah, bawang putih, cabe merah besar, cabe rawit, j...","[Bersihkan ikan, campur dengan air, asam jawa, garam dan kunyit, kemudian masak hingga ikan matang., Ambil bagian dagingnya kemudian suwir suwir, ...","Ikan Suwir Pedas. Bahan: 1 ekor Baby Tuna, 1 sdm Asam Jawa, 1 sdt Garam, 1/2 sdt Kunyit Bubuk, 500 ml Air, 1 lembar Daun Salam, 1 lembar Daun Jeru...",16,5,696
1,Minyak Ayam Bawang,"[100 g kulit ayam, 12 siung bawang merah, 4 siung bawang putih, 250 ml minyak goreng]","100 g kulit ayam, 12 siung bawang merah, 4 siung bawang putih, 250 ml minyak goreng","[{'raw': '100 g kulit ayam', 'quantity': '100', 'unit': 'g', 'name': 'kulit ayam'}, {'raw': '12 siung bawang merah', 'quantity': '12', 'unit': 'si...","[kulit ayam, bawang merah, bawang putih, minyak goreng]","[Rebus kulit ayam hingga air mendidih, angkat., Blender kulit ayam rebus, bawang merah, dan bawang putih. Tambahkan sedikit minyak sebagai pelarut...","Minyak Ayam Bawang. Bahan: 100 g kulit ayam, 12 siung bawang merah, 4 siung bawang putih, 250 ml minyak goreng",4,4,387
2,Puyunghai rolade tahu tuna,"[2 butir telur ayam, 2 buah rolade, 1 tahu tuna, 2 sdm tepung serba guna, 1/4 sdt garam, secukupnya Minyak goreng, Bahan saos:, 1 siung bawang put...","2 butir telur ayam, 2 buah rolade, 1 tahu tuna, 2 sdm tepung serba guna, 1/4 sdt garam, secukupnya Minyak goreng, Bahan saos:, 1 siung bawang puti...","[{'raw': '2 butir telur ayam', 'quantity': '2', 'unit': 'butir', 'name': 'telur ayam'}, {'raw': '2 buah rolade', 'quantity': '2', 'unit': 'buah', ...","[telur ayam, rolade, tahu tuna, tepung serba guna, garam, minyak goreng, bawang putih, saos, kecap manis, air, garam]","[Potong dadu rolade dan tahu tuna., Campurkan semua bahan lalu kocok., Setelah siap, kemudian goreng dengan api kecil., Buat saos, tumis bawang pu...","Puyunghai rolade tahu tuna. Bahan: 2 butir telur ayam, 2 buah rolade, 1 tahu tuna, 2 sdm tepung serba guna, 1/4 sdt garam, secukupnya Minyak goren...",12,5,398


## Catatan & Langkah Selanjutnya

1. **Cek hasil `ingredients_parsed` dan `ingredients_name_only`** pada dataset hasil preprocessing ini — kirim ke saya beberapa contoh yang parsingnya kurang tepat (kalau ada), supaya regex di Bagian 7 bisa disempurnakan sebelum lanjut ke tahap embedding.
2. Setelah preprocessing ini fix, tahap selanjutnya: encode `embedding_text` pakai SBERT (`LazarusNLP/all-indo-e5-small-v4`), lalu index ke ChromaDB dengan metadata `title`, `ingredients_clean`, `ingredients_name_only`, `steps` (supaya bisa dipakai lagi saat menampilkan hasil & menghitung skor efisiensi bahan, tanpa perlu decode ulang dari embedding).
3. `ingredients_name_only` adalah field kunci untuk algoritma reranking efisiensi bahan nanti — dari situ kamu bisa hitung berapa bahan resep yang match dengan input user, dan berapa yang tidak dimiliki user.